## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-5-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings) #Reads existing data from DB

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='3ee281e7-d777-422a-a2c1-fb57f6115882', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery isn’t a single person or thing. It can refer to many things depending on the context. Could you share a bit more about what you’re asking? For now, here are some common possibilities:\n\n- As a given name: Avery is a unisex first name (used for both boys and girls) and also a surname. It has English origins and is used widely in the U.S. today.\n  - Notable people: Avery Brooks (actor from Star Trek: Deep Space Nine), Avery Johnson (former NBA player and coach).\n  - Fictional character: Avery Jessup, a character on the TV show 30 Rock.\n\n- Places: \n  - Avery Island in Louisiana (famous for Tabasco sauce).\n  - Avery County in North Carolina.\n\n- Companies/brands:\n  - Avery Dennison, a large packaging and labeling company.\n  - Avery-brand labels and stationery (popular for printers).\n\nIf you meant a specific Avery (a person, a character in a book/show, or a place), tell me and I can give a precise answer.', additional_kwargs={'refusal': None}, response_m

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs) #joining them all 
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the co-founder and CEO of Insurellm, a San Francisco-based Insurance Tech company. Key details:\n\n- Born March 15, 1985; current location: San Francisco.\n- Salary: $225,000.\n- Insurellm career: Co-Founder & CEO since 2015; helped position Insurellm as a leading player in insurance tech. She’s known for innovative leadership, risk management, and driving market growth.\n- Leadership and initiatives: active in leadership development, diversity and inclusion in hiring, improving work-life balance, and community outreach (notably financial literacy programs).\n- Recognitions: widely regarded in industry circles; 2018 and 2023 ratings show strong performance, with 2021 labeled “Exceptional.”\n- Earlier role: Senior Product Manager at Innovate Insurance Solutions (2013–2015).\n\nIf you want a shorter one-sentence bio or a version for a bio sheet, I can tailor it.'

## What could possibly come next? 😂

In [13]:
gr.ChatInterface(answer_question).launch()

c:\Users\sunil\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!